In [1]:
!pip install unbabel-comet
!pip install openai==0.28

In [2]:
from comet import download_model, load_from_checkpoint
from datasets import load_dataset
import openai
import pandas as pd

In [8]:
prompt = f'''
a text in {{src}} is given to you please translate it to {{des}} without providing anything else
# text
{{text}}
'''

openai.api_key = 'Your-api'
# comment the below line if you use original OPENAI api
openai.api_base = 'https://api.avalai.ir/v1'

In [4]:
model_path = download_model("Unbabel/wmt22-comet-da")
model = load_from_checkpoint(model_path)
dataset = load_dataset("wmt14", "de-en")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

hparams.yaml:   0%|          | 0.00/567 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

de-en/validation-00000-of-00001.parquet:   0%|          | 0.00/474k [00:00<?, ?B/s]

de-en/test-00000-of-00001.parquet:   0%|          | 0.00/509k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4508785 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3003 [00:00<?, ? examples/s]

In [18]:
def get_comet_score(src, mt, ref):
    return model.predict([{
        "src": src,
        "mt": mt,
        "ref": ref}])['system_score']

def get_AI_answer(prompt):
    response = openai.ChatCompletion.create(
        model="gemma-3-12b-it",
        messages=[
            {"role": "user", "content": prompt}
        ])
    result = response['choices'][0]['message']['content'].strip().lower()
    return result

def append_record_to_excel(file_path,src,mt,ref,score):
    new_record = {
            'id': id,
            'Source': src,
            'Machine_translation': mt,
            'Reference': ref,
            'Score': score
            }

    new_record_df = pd.DataFrame([new_record])
    try:
        existing_df = pd.read_excel(file_path)
        updated_df = pd.concat([existing_df, new_record_df], ignore_index=True)
    except FileNotFoundError:
        updated_df = new_record_df

    updated_df.to_excel(file_path, index=False)

In [19]:
# en -> de
for i in range(3003):
    src = dataset['test'][i]['translation']['en']
    mt = get_AI_answer(prompt.format(src='en', des='de', text=src))
    ref = dataset['test'][i]['translation']['de']
    score = get_comet_score(src, mt, ref)
    append_record_to_excel('en2de_evaluation.xlsx',src,mt,ref,score)
    print(i)
    print('===================================================')

INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
Predicting DataLoader 0: 100%|██████████| 1/1 [00:01<00:00,  1.07s/it]


0


INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
Predicting DataLoader 0: 100%|██████████| 1/1 [00:02<00:00,  2.70s/it]


1


INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
Predicting DataLoader 0: 100%|██████████| 1/1 [00:01<00:00,  1.20s/it]


...

INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
Predicting DataLoader 0: 100%|██████████| 1/1 [00:02<00:00,  2.92s/it]


13


INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
Predicting DataLoader 0: 100%|██████████| 1/1 [00:01<00:00,  1.55s/it]


14


APIError: HTTP code 502 from API (<!DOCTYPE html>
<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->
<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->
<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->
<head>


<title>api.avalai.ir | 502: Bad gateway</title>
<meta charset="UTF-8" />
<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />
<meta http-equiv="X-UA-Compatible" content="IE=Edge" />
<meta name="robots" content="noindex, nofollow" />
<meta name="viewport" content="width=device-width,initial-scale=1" />
<link rel="stylesheet" id="cf_styles-css" href="/cdn-cgi/styles/main.css" />


</head>
<body>
<div id="cf-wrapper">
    <div id="cf-error-details" class="p-0">
        <header class="mx-auto pt-10 lg:pt-6 lg:px-8 w-240 lg:w-full mb-8">
            <h1 class="inline-block sm:block sm:mb-2 font-light text-60 lg:text-4xl te

In [17]:
src = dataset['train'][14]['translation']['de']
mt = dataset['train'][14]['translation']['en']
ref = dataset['train'][14]['translation']['en']

print (get_comet_score(src, mt, ref))

INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
Predicting DataLoader 0: 100%|██████████| 1/1 [00:01<00:00,  1.30s/it]

Prediction({'scores': [0.968877375125885], 'system_score': 0.968877375125885})


In [4]:
data = [
    {
        "src": "Dem Feuer konnte Einhalt geboten werden",
        "mt": "The fire could be stopped",
        "ref": "They were able to control the fire."
    }
]
model_output = model.predict(data)
print (model_output)

INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
Predicting DataLoader 0: 100%|██████████| 1/1 [00:01<00:00,  1.20s/it]

Prediction({'scores': [0.8385583162307739], 'system_score': 0.8385583162307739})
